In [2]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
 
# ============================================================
# STEP 1: LOAD DATA & BUILD VOCABULARY
# ============================================================
df = pd.read_csv('train-labels.csv')
print(df.head())
 
all_labels = df['text'].astype(str).tolist()
unique_chars = sorted(set(''.join(all_labels)))
 
# Index 0 is RESERVED for CTC blank token — characters start at 1
char_to_idx = {char: idx + 1 for idx, char in enumerate(unique_chars)}
idx_to_char = {idx + 1: char for idx, char in enumerate(unique_chars)}
 
print(f"Vocabulary size: {len(char_to_idx)}")
print("Characters:", ''.join(unique_chars))
 
 

   Unnamed: 0        image    text
0           0  train-0.png  BU522X
1           1  train-1.png  XQ8NE2
2           2  train-2.png  DTZD3E
3           3  train-3.png  SM424H
4           4  train-4.png  6YVTQR
Vocabulary size: 38
Characters: +-.0123456789ABCDEFGHJKMNPQRSTUVWXYZar


In [3]:
# ============================================================
# STEP 2: DATASET WITH IMPROVED PREPROCESSING
# ============================================================
IMG_WIDTH  = 128
IMG_HEIGHT = 32
 
def preprocess_image(img_path, img_width=IMG_WIDTH, img_height=IMG_HEIGHT):
    """
    Improved preprocessing:
    - CLAHE for local contrast enhancement (helps with noisy/dark images)
    - Resize keeping aspect ratio, then pad
    - Normalize to [0, 1]
    """
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Cannot load image: {img_path}")
 
    # CLAHE: boosts local contrast, especially useful for distorted/dark images
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    img = clahe.apply(img)
 
    # Resize to target dimensions
    img = cv2.resize(img, (img_width, img_height))
 
    # Normalize
    img = img.astype(np.float32) / 255.0
 
    # Shape: [1, H, W]
    return torch.from_numpy(img).unsqueeze(0)
 

In [4]:
class OCRDataset(Dataset):
    def __init__(self, dataframe, img_folder, char_to_idx):
        self.dataframe   = dataframe
        self.img_folder  = img_folder
        self.char_to_idx = char_to_idx
 
    def __len__(self):
        return len(self.dataframe)
 
    def __getitem__(self, idx):
        img_name   = str(self.dataframe.iloc[idx]['image'])
        text_label = str(self.dataframe.iloc[idx]['text'])
        img_path   = os.path.join(self.img_folder, img_name)
 
        img_tensor = preprocess_image(img_path)
 
        label_encoded = [self.char_to_idx[c] for c in text_label if c in self.char_to_idx]
        label_tensor  = torch.tensor(label_encoded, dtype=torch.long)
 
        return img_tensor, label_tensor

In [5]:
def custom_collate(batch):
    images, labels, target_lengths = [], [], []
    for img, label in batch:
        images.append(img)
        labels.extend(label.tolist())
        target_lengths.append(len(label))
    images         = torch.stack(images, 0)
    labels         = torch.tensor(labels, dtype=torch.long)
    target_lengths = torch.tensor(target_lengths, dtype=torch.long)
    return images, labels, target_lengths
 

In [6]:
# STEP 3: MODEL
# ============================================================
class CRNN(nn.Module):
    """
    CRNN with verified output dimensions.
 
    Input:  [B, 1, 32, 128]
    After CNN:
      Conv1 + Pool(2,2) → [B, 32, 16, 64]
      Conv2 + Pool(2,2) → [B, 64,  8, 32]
      Conv3 + Pool(2,1) → [B, 128, 4, 32]   ← width stays 32!
    Reshape to sequence: [B, 32, 128*4] = [B, 32, 512]
    Linear → [B, 32, 256]
    BiLSTM → [B, 32, 512]
    Linear → [B, 32, vocab_size+1]
    Permute → [32, B, vocab_size+1]   ← T=32 time steps for CTC
    """
    def __init__(self, vocab_size):
        super(CRNN, self).__init__()
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),          # → [B, 32, 16, 64]
 
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),          # → [B, 64, 8, 32]
 
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)), # → [B, 128, 4, 32]
        )
 
        # After CNN: H=4, C=128 → feature per time-step = 128*4 = 512
        self.cnn_feature_size = 128 * 4   # = 512
 
        self.linear1 = nn.Linear(self.cnn_feature_size, 256)
        self.dropout  = nn.Dropout(0.3)
        self.lstm     = nn.LSTM(256, 256, bidirectional=True, num_layers=2,
                                batch_first=True, dropout=0.3)
        self.linear2  = nn.Linear(256 * 2, vocab_size + 1)
 
    def forward(self, x):
        # x: [B, 1, 32, 128]
        conv_out = self.cnn(x)                              # [B, 128, 4, 32]
 
        batch, channels, height, width = conv_out.size()
        # Merge channels and height → feature vector per column
        conv_out = conv_out.view(batch, channels * height, width)  # [B, 512, 32]
        conv_out = conv_out.permute(0, 2, 1)                       # [B, 32, 512]
 
        linear_out = F.relu(self.linear1(conv_out))         # [B, 32, 256]
        linear_out = self.dropout(linear_out)
 
        lstm_out, _ = self.lstm(linear_out)                 # [B, 32, 512]
        output = self.linear2(lstm_out)                     # [B, 32, vocab+1]
        output = F.log_softmax(output, dim=2)
        output = output.permute(1, 0, 2)                    # [32, B, vocab+1]
        return output
 
    def get_output_width(self):
        """Returns T (time steps) — needed for input_lengths in CTC."""
        # Width after CNN: 128 → /2 → /2 → /1 (last pool stride=1 on W) = 32
        return 32

In [7]:
# STEP 4: TRAINING
# ============================================================
IMAGE_FOLDER = r"C:\Users\Vishesh Kumar\Downloads\cig_ps\cig_ps\train_images"
 
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
 
train_dataset = OCRDataset(train_df, IMAGE_FOLDER, char_to_idx)
val_dataset   = OCRDataset(val_df,   IMAGE_FOLDER, char_to_idx)
 
# Smaller batch size → more stable CTC gradient updates
BATCH_SIZE    = 32
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=custom_collate, num_workers=0)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                           collate_fn=custom_collate, num_workers=0)
 
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
 
vocab_size = len(char_to_idx)
model      = CRNN(vocab_size)
 
# FIX: compute the ACTUAL sequence length T from the model architecture
T = model.get_output_width()   # = 32
print(f"CTC input_lengths (T) = {T}")
 
criterion = nn.CTCLoss(blank=0, zero_infinity=True, reduction='mean')
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                  patience=3, factor=0.5)
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Training on: {device}")
 
num_epochs    = 20
best_val_loss = float('inf')
 
for epoch in range(num_epochs):
    # --- TRAIN ---
    model.train()
    running_train_loss = 0.0
 
    for images, labels, target_lengths in train_loader:
        images, labels, target_lengths = (images.to(device),
                                          labels.to(device),
                                          target_lengths.to(device))
        batch_size = images.size(0)
 
        optimizer.zero_grad()
        outputs = model(images)   # [T, B, vocab+1]
 
        # FIX: input_lengths must match the actual T output by the model
        input_lengths = torch.full((batch_size,), fill_value=T,
                                   dtype=torch.long, device=device)
 
        loss = criterion(outputs, labels, input_lengths, target_lengths)
 
        loss.backward()
 
        # Gradient clipping: prevents exploding gradients with CTC
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
 
        optimizer.step()
        running_train_loss += loss.item()
 
    avg_train_loss = running_train_loss / len(train_loader)
 
    # --- VALIDATE ----
    model.eval()
    running_val_loss = 0.0
 
    with torch.no_grad():
        for images, labels, target_lengths in val_loader:
            images, labels, target_lengths = (images.to(device),
                                              labels.to(device),
                                              target_lengths.to(device))
            batch_size = images.size(0)
            outputs    = model(images)
 
            input_lengths = torch.full((batch_size,), fill_value=T,
                                       dtype=torch.long, device=device)
            loss = criterion(outputs, labels, input_lengths, target_lengths)
            running_val_loss += loss.item()
 
    avg_val_loss = running_val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
 
    print(f"Epoch [{epoch+1:>2}/{num_epochs}] | "
          f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
 
    if avg_val_loss < best_val_loss:
        print(f"  🌟 New best! Val loss {best_val_loss:.4f} → {avg_val_loss:.4f}. Saving...")
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_crnn_model.pth")
 
    print("-" * 60)

Train batches: 500 | Val batches: 125
CTC input_lengths (T) = 32
Training on: cpu
Epoch [ 1/20] | Train Loss: 3.7551 | Val Loss: 3.6401
  🌟 New best! Val loss inf → 3.6401. Saving...
------------------------------------------------------------
Epoch [ 2/20] | Train Loss: 3.6275 | Val Loss: 3.6206
  🌟 New best! Val loss 3.6401 → 3.6206. Saving...
------------------------------------------------------------
Epoch [ 3/20] | Train Loss: 3.5851 | Val Loss: 3.4284
  🌟 New best! Val loss 3.6206 → 3.4284. Saving...
------------------------------------------------------------
Epoch [ 4/20] | Train Loss: 3.0429 | Val Loss: 2.6824
  🌟 New best! Val loss 3.4284 → 2.6824. Saving...
------------------------------------------------------------
Epoch [ 5/20] | Train Loss: 2.2201 | Val Loss: 1.6049
  🌟 New best! Val loss 2.6824 → 1.6049. Saving...
------------------------------------------------------------
Epoch [ 6/20] | Train Loss: 1.2167 | Val Loss: 0.8270
  🌟 New best! Val loss 1.6049 → 0.8270. Sa

In [ ]:
# ============================================================
def decode_predictions(predictions, idx_to_char):
    """
    CTC greedy decoder:
    1. Take argmax at each time step
    2. Collapse repeated characters
    3. Remove blank tokens (index 0)
    """
    _, max_indices = torch.max(predictions, dim=2)  # [T, B]
    max_indices    = max_indices.permute(1, 0)       # [B, T]
 
    decoded = []
    for sequence in max_indices:
        chars      = []
        prev_idx   = -1
        for idx in sequence:
            idx = idx.item()
            if idx != 0 and idx != prev_idx:          # not blank, not repeat
                if idx in idx_to_char:
                    chars.append(idx_to_char[idx])
            prev_idx = idx
        decoded.append(''.join(chars))
    return decoded
 
 
# ============================================================
# STEP 6: GENERATE PREDICTIONS
# ============================================================
TEST_FOLDER = r"C:\Users\Vishesh Kumar\Downloads\cig_ps\cig_ps\test_images"
 
print("\nLoading best model for predictions...")
model.load_state_dict(torch.load("best_crnn_model.pth", weights_only=True))
model.eval()
 
results = []
print("Predicting on test set...")
 
with torch.no_grad():
    for img_name in sorted(os.listdir(TEST_FOLDER)):
        if not img_name.lower().endswith('.png'):
            continue
 
        img_path   = os.path.join(TEST_FOLDER, img_name)
        img_tensor = preprocess_image(img_path)                    # [1, 32, 128]
        img_tensor = img_tensor.unsqueeze(0).to(device)            # [1, 1, 32, 128]
 
        outputs    = model(img_tensor)                             # [T, 1, vocab+1]
        pred_text  = decode_predictions(outputs, idx_to_char)[0]
        results.append({'image': img_name, 'prediction': pred_text})
        print(f"  {img_name} → {pred_text}")
 
submission_df = pd.DataFrame(results)
csv_filename  = "submission_Vishesh_Kumar_24112114.csv"
submission_df.to_csv(csv_filename, index=False)
print(f"\nDone! Saved {len(results)} predictions to: {csv_filename}")
 


Loading best model for predictions...
Predicting on test set...
  test-0.png → QVTQ8A
  test-1.png → 7PSW9D
  test-10.png → 7DUP98
  test-100.png → 75Z4WT
  test-1000.png → QAK27V
  test-1001.png → R6MERY
  test-1002.png → CHXX67
  test-1003.png → 9NV2WP
  test-1004.png → F56TDZ
  test-1005.png → FFTFRX
  test-1006.png → ES7T4B
  test-1007.png → ZX4G8H
  test-1008.png → WZA48S
  test-1009.png → WNDUV3
  test-101.png → 32KDUK
  test-1010.png → QMX9BV
  test-1011.png → 45TZR9
  test-1012.png → 4XGTBK
  test-1013.png → PCCRK7
  test-1014.png → 2N2J95
  test-1015.png → 5ZK5TR
  test-1016.png → HUESAY
  test-1017.png → GV7VH9
  test-1018.png → F25TPE
  test-1019.png → B9SAXS
  test-102.png → N6A65C
  test-1020.png → UNKP8
  test-1021.png → 3CV64J
  test-1022.png → TQZHHY
  test-1023.png → ZPHFAT
  test-1024.png → X3M8WW
  test-1025.png → DZERHN
  test-1026.png → HF7ZVT
  test-1027.png → V8NQJK
  test-1028.png → EZ8BR9
  test-1029.png → B4M2T3
  test-103.png → DUPHYM
  test-1030.png → JN7BV

In [ ]:
print("\n--- Sanity check on 5 training images ---")
model.eval()
sample_df = train_df.sample(5, random_state=0).reset_index(drop=True)
 
with torch.no_grad():
    for i in range(len(sample_df)):
        img_name   = sample_df.iloc[i]['image']
        true_text  = sample_df.iloc[i]['text']
        img_path   = os.path.join(IMAGE_FOLDER, img_name)
        img_tensor = preprocess_image(img_path).unsqueeze(0).to(device)
        outputs    = model(img_tensor)
        pred_text  = decode_predictions(outputs, idx_to_char)[0]
        status     = "✅" if pred_text == true_text else "❌"
        print(f"  {status} True: {true_text:<15} Predicted: {pred_text}")
 


--- Sanity check on 5 training images ---
  ✅ True: RZBGUH          Predicted: RZBGUH
  ✅ True: JVZYRN          Predicted: JVZYRN
  ✅ True: HRDEJR          Predicted: HRDEJR
  ❌ True: QGWSGG          Predicted: QCWSGG
  ✅ True: WEPMHP          Predicted: WEPMHP
